# Hybrid Probabilistic + FSM-Based Sequence Modeling for Document Boundary Detection

This notebook demonstrates a hybrid pipeline for detecting document boundaries (e.g., splitting a stream of scanned pages into separate multi-page documents). It combines:
1. **Probabilistic Page Classifier**: Predicts the likelihood of a page being a **First page (F)** or a **Next page (N)** based on page-level features (text length, presence of key terms like 'invoice', 'date', 'page 1 of...').
2. **Finite State Machine (FSM) Decoder**: Restricts sequence transitions to guarantee valid document flows (e.g., you cannot have a 'Next' page without an initial 'First' page, and enforcing minimum/maximum page length rules).

In [1]:
import numpy as np
import pandas as pd

# Set random seed for reproducibility
np.random.seed(42)

# 1. Generate Dummy Page-Level Features for 30 pages
# Assume a document stream with true boundary indices: [0, 5, 12, 18, 25]
# True classes: F = First Page, N = Next/Intermediate Page

true_boundaries = [0, 5, 12, 18, 25]
total_pages = 30
y_true = ['F' if i in true_boundaries else 'N' for i in range(total_pages)]

# Features:
# - has_header_keywords: highly correlated with 'F' (0.8 vs 0.1)
# - page_number_indicator: if page mentions "1" or "first" (0.9 vs 0.05)
# - visual_signature_score: scalar representation of cover-like visual features

data = []
for i, label in enumerate(y_true):
    if label == 'F':
        has_header = np.random.binomial(1, 0.85)
        is_page_one = np.random.binomial(1, 0.90)
        visual_score = np.random.normal(0.8, 0.15)
    else:
        has_header = np.random.binomial(1, 0.15)
        is_page_one = np.random.binomial(1, 0.05)
        visual_score = np.random.normal(0.2, 0.2)

    data.append({
        'page_index': i,
        'has_header': has_header,
        'is_page_one': is_page_one,
        'visual_score': max(0.0, min(1.0, visual_score)),
        'true_label': label
    })

df = pd.DataFrame(data)
display(df.head(10))

,page_index,has_header,is_page_one,visual_score,true_label
0,0,1,0,0.897153,F
1,1,0,0,0.504606,N
2,2,0,0,0.515843,N
3,3,0,1,0.353487,N
4,4,0,0,0.107316,N
5,5,1,1,0.730141,F
6,6,0,0,0.000000,N
7,7,0,0,0.262849,N
8,8,0,0,0.213506,N
9,9,0,0,0.000000,N


## Phase 1: Probabilistic Sequence Scoring

We will train/define a quick heuristic function that outputs the probability $P(F)$ that a page is a boundary start.

In [2]:
def predict_start_probability(row):
    # A simple logistic-like heuristic or mock classifier output
    score = 0.3 * row['has_header'] + 0.4 * row['is_page_one'] + 0.3 * row['visual_score']
    return score

df['P_First'] = df.apply(predict_start_probability, axis=1)
df['P_Next'] = 1.0 - df['P_First']

display(df[['page_index', 'true_label', 'P_First', 'P_Next']].head(10))

,page_index,true_label,P_First,P_Next
0,0,F,0.569146,0.430854
1,1,N,0.151382,0.848618
2,2,N,0.154753,0.845247
3,3,N,0.506046,0.493954
4,4,N,0.032195,0.967805
5,5,F,0.919042,0.080958
6,6,N,0.000000,1.000000
7,7,N,0.078855,0.921145
8,8,N,0.064052,0.935948
9,9,N,0.000000,1.000000


## Phase 2: Finite State Machine (FSM) Constraint Decoding

We implement a Viterbi-like decoding pass or a greedy state machine that maps page probabilities to structurally valid sequences.

**Rules:**
- The very first page (index 0) must be a 'F' (First).
- An 'N' cannot follow another sequence without starting with 'F'.
- Optional: Max document length constraint (e.g., auto-trigger 'F' if current document exceeds 8 pages).

In [3]:
def fsm_decode(probabilities, max_doc_len=8):
    """
    Decodes a stream of page probabilities into 'F' (First) and 'N' (Next)
    states while enforcing FSM structural constraints.
    """
    decoded_states = []
    current_doc_len = 0

    for i, p_first in enumerate(probabilities):
        # Rule 1: First page of stream must start a document
        if i == 0:
            state = 'F'
        # Rule 2: Hard ceiling on max document length
        elif current_doc_len >= max_doc_len:
            state = 'F'
        else:
            # Decision threshold with safety margins
            if p_first > 0.45:
                state = 'F'
            else:
                state = 'N'

        if state == 'F':
            current_doc_len = 1
        else:
            current_doc_len += 1

        decoded_states.append(state)

    return decoded_states

df['predicted_label'] = fsm_decode(df['P_First'].values)

# Evaluate accuracy
correct = (df['true_label'] == df['predicted_label']).sum()
print(f"Sequence Accuracy: {correct}/{len(df)} ({correct/len(df)*100:.2f}%)")
display(df[['page_index', 'true_label', 'predicted_label', 'P_First']])

Sequence Accuracy: 29/30 (96.67%)


,page_index,true_label,predicted_label,P_First
0,0,F,F,0.569146
1,1,N,N,0.151382
2,2,N,N,0.154753
3,3,N,F,0.506046
4,4,N,N,0.032195
5,5,F,F,0.919042
6,6,N,N,0.000000
7,7,N,N,0.078855
8,8,N,N,0.064052
9,9,N,N,0.000000
